# Weather Data EDA

## 1. Load Raw Data

In [41]:
#Create a project root for accessing other files
import sys
from pathlib import Path

Project_root = Path.cwd().parent
sys.path.append(str(Project_root))

#Importing the existing code
from src.config import Config

#Find raw JSON
import json

json_files = list(Config.raw_data_path.glob("*.json"))

latest_file = max(json_files, key=lambda file: file.stat().st_mtime)

with open(latest_file, "r", encoding="utf-8")as file:
    raw_data = json.load(file)

## 2. Normalize Raw JSON

In [42]:

from src.transformation import normalize_json

#Use the normalize function
df = normalize_json(raw_data)

import pandas as pd


In [43]:
#Renaming columns after normalize and explode for better reading
df.columns = df.columns.str.split('.').str[-1]
df = df.rename(columns={
        "areatype": "area_type",
        "lat": "latitude",
        "lon": "longitude",
        "time": "forecast_time",
        "cloudhigh": "cloud_high",
        "cloudmed": "cloud_medium",
        "cloudlow": "cloud_low",
        "cond": "condition",
        "rh": "relative_humidity",
        "slp": "sea_level_pressure",
        "tc": "temperature",
        "tc_max": "temperature_max",
        "tc_min": "temperature_min",
        "wd10m": "wind_direction",
        "ws10m": "wind_speed"
    })
print(df.columns)

Index(['province', 'area_type', 'name', 'tambon', 'region', 'geocode',
       'amphoe', 'latitude', 'longitude', 'forecast_time', 'cloud_high',
       'cloud_low', 'cloud_medium', 'condition', 'rain', 'relative_humidity',
       'sea_level_pressure', 'temperature', 'temperature_max',
       'temperature_min', 'wind_direction', 'wind_speed'],
      dtype='str')


## 3. Pre-Transformation Data Quality Assessment

### 3.1 Dataset Structure

In [44]:
df.shape

(98, 22)

In [45]:
df.head()

,province,area_type,name,tambon,region,geocode,amphoe,latitude,longitude,forecast_time,...,cloud_medium,condition,rain,relative_humidity,sea_level_pressure,temperature,temperature_max,temperature_min,wind_direction,wind_speed
0,กระบี่,province,กระบี่,None,S,81,None,8.062061,98.918394,2026-08-23T00:00:00+07:00,...,41.33,3,0.2,84.25,1011.64,27.88,30.81,24.76,283.29,8.39
1,กระบี่,province,กระบี่,None,S,81,None,8.062061,98.918394,2026-08-24T00:00:00+07:00,...,75.63,3,14.6,84.07,1011.16,27.79,32.30,25.73,264.94,9.93
2,กระบี่,province,กระบี่,None,S,81,None,8.062061,98.918394,2026-08-25T00:00:00+07:00,...,44.63,3,6.0,86.42,1010.89,27.64,30.46,25.50,255.93,9.12
3,กระบี่,province,กระบี่,None,S,81,None,8.062061,98.918394,2026-08-26T00:00:00+07:00,...,20.63,4,6.1,83.27,1011.17,27.93,30.09,26.52,248.05,8.97
4,กระบี่,province,กระบี่,None,S,81,None,8.062061,98.918394,2026-08-27T00:00:00+07:00,...,25.88,5,2.4,84.11,1011.00,27.66,30.74,25.46,238.31,8.64


In [46]:
df.columns.tolist()
#list all columns

['province',
 'area_type',
 'name',
 'tambon',
 'region',
 'geocode',
 'amphoe',
 'latitude',
 'longitude',
 'forecast_time',
 'cloud_high',
 'cloud_low',
 'cloud_medium',
 'condition',
 'rain',
 'relative_humidity',
 'sea_level_pressure',
 'temperature',
 'temperature_max',
 'temperature_min',
 'wind_direction',
 'wind_speed']

### 3.2 Data Types

In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   province            98 non-null     str    
 1   area_type           98 non-null     str    
 2   name                98 non-null     str    
 3   tambon              0 non-null      object 
 4   region              98 non-null     str    
 5   geocode             98 non-null     str    
 6   amphoe              0 non-null      object 
 7   latitude            98 non-null     float64
 8   longitude           98 non-null     float64
 9   forecast_time       98 non-null     str    
 10  cloud_high          98 non-null     float64
 11  cloud_low           98 non-null     float64
 12  cloud_medium        98 non-null     float64
 13  condition           98 non-null     int64  
 14  rain                98 non-null     float64
 15  relative_humidity   98 non-null     float64
 16  sea_level_pressure  9

In [48]:
df.dtypes
#We found here that the "forecast_time" is still a string

province                  str
area_type                 str
name                      str
tambon                 object
region                    str
geocode                   str
amphoe                 object
latitude              float64
longitude             float64
forecast_time             str
cloud_high            float64
cloud_low             float64
cloud_medium          float64
condition               int64
rain                  float64
relative_humidity     float64
sea_level_pressure    float64
temperature           float64
temperature_max       float64
temperature_min       float64
wind_direction        float64
wind_speed            float64
dtype: object

### 3.3 Missing Values

In [49]:
df.isna().sum()
#The data doesn't contain specific tambon and amphoe.

province               0
area_type              0
name                   0
tambon                98
region                 0
geocode                0
amphoe                98
latitude               0
longitude              0
forecast_time          0
cloud_high             0
cloud_low              0
cloud_medium           0
condition              0
rain                   0
relative_humidity      0
sea_level_pressure     0
temperature            0
temperature_max        0
temperature_min        0
wind_direction         0
wind_speed             0
dtype: int64

In [50]:
#Since tambon and amphoe is intentionally left NULL and it is not important for further analysis so we can remove it.
df = df.drop(columns=['tambon','amphoe'])

In [51]:
df.columns.tolist()

['province',
 'area_type',
 'name',
 'region',
 'geocode',
 'latitude',
 'longitude',
 'forecast_time',
 'cloud_high',
 'cloud_low',
 'cloud_medium',
 'condition',
 'rain',
 'relative_humidity',
 'sea_level_pressure',
 'temperature',
 'temperature_max',
 'temperature_min',
 'wind_direction',
 'wind_speed']

### 3.4 Duplicate Records

In [52]:
df.duplicated().sum()

np.int64(0)

### 3.5 Numerical Value Ranges

In [53]:
numeric_col = df.select_dtypes(include=["float64"])

In [54]:
numeric_col.describe()

,latitude,longitude,cloud_high,cloud_low,cloud_medium,rain,relative_humidity,sea_level_pressure,temperature,temperature_max,temperature_min,wind_direction,wind_speed
count,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000
mean,7.946897,99.829879,38.064286,66.049490,45.680918,6.797959,80.548776,1010.597143,27.794898,31.789898,25.109388,249.250204,9.351224
std,1.215173,1.061427,17.341285,19.367595,19.784431,11.279641,6.107017,0.770820,0.710585,2.443437,1.122460,59.030978,2.604320
min,6.428561,98.385376,1.880000,9.250000,2.880000,0.000000,58.120000,1008.070000,26.030000,28.000000,21.850000,20.520000,3.990000
25%,6.868660,98.918394,23.630000,55.907500,31.567500,0.450000,76.427500,1010.152500,27.350000,30.247500,24.392500,247.870000,7.577500
50%,7.754199,99.786312,41.520000,70.005000,43.130000,2.300000,81.630000,1010.900000,27.780000,31.040000,25.155000,257.300000,9.120000
75%,8.442509,100.596251,51.437500,79.577500,60.722500,8.500000,85.000000,1011.130000,28.347500,32.665000,25.842500,269.277500,10.807500
max,10.492748,101.822871,72.250000,98.130000,89.380000,64.900000,91.000000,1011.730000,29.460000,42.750000,27.640000,357.710000,15.830000


### 3.6 Categorical Values

In [55]:
df["province"].value_counts()

province
กระบี่           7
ชุมพร            7
ตรัง             7
นครศรีธรรมราช    7
นราธิวาส         7
ปัตตานี          7
พังงา            7
พัทลุง           7
ภูเก็ต           7
ยะลา             7
ระนอง            7
สงขลา            7
สตูล             7
สุราษฎร์ธานี     7
Name: count, dtype: int64

In [56]:
df["condition"].value_counts()

condition
3    34
4    32
5    17
1    11
2     4
Name: count, dtype: int64

In [57]:
df["condition"].nunique()

5

In [58]:
df["condition"].unique()

array([3, 4, 5, 1, 2])

### 3.7 Temporal Data

In [59]:
df["forecast_time"] = pd.to_datetime(df["forecast_time"])

In [60]:
df["forecast_time"].head()

0   2026-08-23 00:00:00+07:00
1   2026-08-24 00:00:00+07:00
2   2026-08-25 00:00:00+07:00
3   2026-08-26 00:00:00+07:00
4   2026-08-27 00:00:00+07:00
Name: forecast_time, dtype: datetime64[us, UTC+07:00]

In [61]:
df["forecast_time"].min()

Timestamp('2026-08-23 00:00:00+0700', tz='UTC+07:00')

In [62]:
df["forecast_time"].max()

Timestamp('2026-08-29 00:00:00+0700', tz='UTC+07:00')

In [63]:
df["forecast_time"].isnull().sum()

np.int64(0)

In [64]:
df["forecast_time"].diff().value_counts()
#It counts how many times each difference happened. Example: There were 84 instances where the next row was exactly 1 day after the previous row and there are 13 instances where the next row was 6 days earlier than the previous row.

forecast_time
1 days     84
-6 days    13
Name: count, dtype: int64

## EDA Summary

- The dataset contains 98 rows and 22 columns after JSON normalization.
- No unexpected missing values were identified. NULL values in `amphoe` and `tambon` are expected because the dataset is retrieved at the province/region level. These fields are intentionally left NULL. So we can remove it.
- No duplicate records were identified.
- Numerical variables have reasonable ranges based on their weather-related meaning.
- Categorical weather conditions contain expected values.
- The `time` column is currently stored as a string and should be converted to datetime during transformation.
- Weather measurement columns are already represented using approproate numeric data types.